# 05. Unión de SISAM al panel semanal

**Propósito.** Agregar la calidad del aire (SISAM) al panel base a nivel semanal. Solo la parte semanal: ya no existe la unión por alerta.

**Entradas:** `datos/datasets_preprocesamiento/panel_semanal_base.csv` y `SISAM/sisam_limpio.csv`.

**Salida:** `datos/datasets_preprocesamiento/panel_semanal_sisam.csv`.

**Qué cambió respecto al Hito 2**
- Se eliminó la unión a nivel de alerta y la ventana móvil por alerta.
- La agregación semanal usa `semana_lunes` y el calendario de 251 semanas.
- Nuevas columnas `dias_sisam` (días con dato en la semana) y `fuente_sisam` (`reanalise`, `pronostico` o `mixta`).
- Una semana con menos de 7 días de dato deja las seis variables en NaN.

In [1]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Entorno: Google Colab (Drive) o ejecucion local
# ---------------------------------------------------------------------------
USAR_DRIVE = True   # True en Colab; False para trabajar con la carpeta local del repositorio
BASE_DRIVE = Path("/content/drive/MyDrive/BIODIVERSITY_GUARD")

if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = BASE_DRIVE
    except ImportError:
        print("No se detecto Google Colab: se usa la raiz local del repositorio.")
        USAR_DRIVE = False

if not USAR_DRIVE:
    # la raiz es la primera carpeta (hacia arriba desde la actual) que contiene "datos"
    _aqui = Path.cwd().resolve()
    BASE = next((p for p in [_aqui, *_aqui.parents] if (p / "datos").is_dir()), None)
    if BASE is None:
        raise FileNotFoundError("No se encontro la carpeta 'datos' subiendo desde " + str(_aqui))

if not BASE.exists():
    raise FileNotFoundError(f"No existe BASE = {BASE}. Verifica la carpeta BIODIVERSITY_GUARD en Drive.")

# ---------------------------------------------------------------------------
# 2. Rutas (todas salen de BASE)
# ---------------------------------------------------------------------------
DIR_PREP = BASE / "datos" / "datasets_preprocesamiento"
DIR_BDQ = DIR_PREP / "BDQueimadas"
DIR_DETER = DIR_PREP / "DETER"
DIR_NASA = DIR_PREP / "NasaPower"
DIR_SISAM = DIR_PREP / "SISAM"
DIR_EDA = BASE / "datos" / "datasets_eda"

# ---------------------------------------------------------------------------
# 3. Constantes comunes (identicas en los 7 notebooks)
# ---------------------------------------------------------------------------
FECHA_INICIO_DATOS = "2022-01-01"
FECHA_FIN_EXTRACCION = "2026-11-01"      # DETER y BDQueimadas
PRIMERA_SEMANA = "2022-01-03"             # primer lunes con semana completa (2022-01-01 es sabado)
ULTIMA_SEMANA = "2026-10-19"              # lunes de la ultima semana completa (termina domingo 2026-10-25)
SATELITE_REFERENCIA = "AQUA_M-T"
ESTADOS_AMAZONIA_LEGAL = ["ACRE", "AMAPÁ", "AMAZONAS", "MARANHÃO", "MATO GROSSO", "PARÁ", "RONDÔNIA", "RORAIMA", "TOCANTINS"]
MAPA_UF = {"ACRE": "AC", "AMAPÁ": "AP", "AMAZONAS": "AM", "MARANHÃO": "MA", "MATO GROSSO": "MT", "PARÁ": "PA", "RONDÔNIA": "RO", "RORAIMA": "RR", "TOCANTINS": "TO"}
CORRECCIONES_MUNICIPIO = {"ELDORADO DOS CARAJAS": "ELDORADO DO CARAJAS", "SANTA ISABEL DO PARA": "SANTA IZABEL DO PARA"}
FECHA_CORTE_CENTROIDES = "2024-12-31"     # los centroides se fijan con focos hasta esta fecha

# ---------------------------------------------------------------------------
# 4. Funciones comunes
# ---------------------------------------------------------------------------
def normalizar_texto(texto):
    # mayusculas, sin tildes ni espacios sobrantes
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("utf-8")


def semana_lunes(fecha):
    # lunes de la semana (lunes a domingo) de cada fecha; recibe una Serie de fechas
    f = pd.to_datetime(fecha).dt.normalize()
    return f - pd.to_timedelta(f.dt.day_of_week, unit="D")


SEMANAS = pd.date_range(PRIMERA_SEMANA, ULTIMA_SEMANA, freq="W-MON")   # calendario de semanas completas
assert len(SEMANAS) == 251, f"El calendario deberia tener 251 semanas y tiene {len(SEMANAS)}"
assert (SEMANAS.day_of_week == 0).all(), "Todas las semanas deben empezar en lunes"


def verificar_entradas(archivos):
    # imprime los archivos de entrada con su tamano y falla si falta alguno
    faltan = []
    for a in map(Path, archivos):
        if a.exists():
            print(f"  OK  {a.relative_to(BASE)}  ({a.stat().st_size / 1e6:,.1f} MB)")
        else:
            faltan.append(a)
    if faltan:
        lista = "\n".join(f"  - {a}" for a in faltan)
        raise FileNotFoundError("Faltan archivos de entrada (copialos a esas carpetas y vuelve a ejecutar):\n" + lista)


print("BASE:", BASE)
print("Calendario:", SEMANAS[0].date(), "a", SEMANAS[-1].date(), f"({len(SEMANAS)} semanas)")

RUTA_PANEL = DIR_PREP / "panel_semanal_base.csv"
RUTA_SISAM = DIR_SISAM / "sisam_limpio.csv"
RUTA_SALIDA = DIR_PREP / "panel_semanal_sisam.csv"
CLAVE = ["municipio_norm", "uf"]
COLS_SISAM = ["pm10_reanalise", "pm2_5_reanalise", "o3_reanalise", "no2_reanalise", "co_reanalise", "so2_reanalise"]
DIAS_MINIMOS = 7   # una semana necesita los 7 dias para calcular el promedio

print("\nArchivos de entrada:")
verificar_entradas([RUTA_PANEL, RUTA_SISAM])

Mounted at /content/drive
BASE: /content/drive/MyDrive/BIODIVERSITY_GUARD
Calendario: 2022-01-03 a 2026-10-19 (251 semanas)

Archivos de entrada:
  OK  datos/datasets_preprocesamiento/panel_semanal_base.csv  (13.0 MB)
  OK  datos/datasets_preprocesamiento/SISAM/sisam_limpio.csv  (112.6 MB)


In [2]:
panel = pd.read_csv(RUTA_PANEL, parse_dates=["semana"])
sisam = pd.read_csv(RUTA_SISAM, parse_dates=["data"])
filas_panel = len(panel)
print("Panel base:", panel.shape)
print("SISAM diario:", sisam.shape, "| rango:", sisam["data"].min().date(), "a", sisam["data"].max().date())

sisam["semana"] = semana_lunes(sisam["data"])
fuera = (~sisam["semana"].isin(SEMANAS)).sum()
print(f"Filas diarias de SISAM fuera del calendario (descartadas): {fuera:,}")
sisam = sisam[sisam["semana"].isin(SEMANAS)]

Panel base: (136293, 15)
SISAM diario: (1427155, 11) | rango: 2022-01-01 a 2026-10-31
Filas diarias de SISAM fuera del calendario (descartadas): 6,470


## Agregación semanal

Por `(semana, municipio_norm, uf)`: promedio de las seis variables, `dias_sisam` (días con dato) y `fuente_sisam`. Si una semana mezcla reanálisis y pronóstico (por ejemplo la semana que cruza el cambio de fuente, del 29 dic 2025 al 4 ene 2026), se marca `"mixta"`.

**Decisión:** una semana con menos de 7 días de dato queda con las variables en NaN, en lugar de promediar pocos días, para que todos los promedios semanales representen el mismo esfuerzo de medición.

In [3]:
def fuente_semana(s):
    u = s.unique()
    return u[0] if len(u) == 1 else "mixta"


sisam["_hay_dato"] = sisam[COLS_SISAM].notna().any(axis=1)
sisam_semanal = (
    sisam.groupby(["semana"] + CLAVE)
    .agg(**{c: (c, "mean") for c in COLS_SISAM},
         dias_sisam=("_hay_dato", "sum"),
         fuente_sisam=("fuente_sisam", fuente_semana))
    .reset_index()
)
sisam_semanal["dias_sisam"] = sisam_semanal["dias_sisam"].astype(int)

incompletas = sisam_semanal["dias_sisam"] < DIAS_MINIMOS
print(f"Semanas-municipio con SISAM: {len(sisam_semanal):,}")
print(f"Semanas con menos de {DIAS_MINIMOS} dias de dato (variables a NaN): {incompletas.sum():,}")
print(sisam_semanal.loc[incompletas, "semana"].dt.date.value_counts().sort_index().head(10))
sisam_semanal.loc[incompletas, COLS_SISAM] = np.nan
print("\nFuente por semana:\n", sisam_semanal["fuente_sisam"].value_counts())

Semanas-municipio con SISAM: 202,955
Semanas con menos de 7 dias de dato (variables a NaN): 0
Series([], Name: count, dtype: int64)

Fuente por semana:
 fuente_sisam
reanalise     168168
pronostico     33978
mixta            809
Name: count, dtype: int64


### Hallazgos de esta etapa

**Qué se observa**
- Hay **202,955** semanas-municipio con SISAM. Por fuente: reanálisis **168,168**, pronóstico **33,978** y mixta **809**.
- Hay **0** semanas con menos de 7 días de dato.
- Se descartaron **6,470** días de SISAM por caer fuera del calendario.

**Por qué se decidió así**
- Las semanas "mixtas" son las que cruzan el cambio de fuente (29 de diciembre de 2025 al 4 de enero de 2026). Se marcan para no presentarlas como reanálisis puro.
- La regla de 7 días evita promediar pocas observaciones. En este caso no tuvo efecto, porque ninguna semana quedó incompleta.

**Qué implica para el modelado**
- **Limitación:** la semana mixta combina dos productos de nivel distinto. `fuente_sisam` permite tratarla aparte.


In [4]:
panel = panel.merge(sisam_semanal, on=["semana"] + CLAVE, how="left", validate="one_to_one")
panel["dias_sisam"] = panel["dias_sisam"].fillna(0).astype(int)

assert len(panel) == filas_panel, f"El merge cambio el numero de filas: {filas_panel} -> {len(panel)}"
assert not panel.duplicated(CLAVE + ["semana"]).any(), "Clave duplicada tras el merge"
assert set(panel["semana"].unique()) == set(SEMANAS)
print("Controles OK. Panel con SISAM:", panel.shape)

Controles OK. Panel con SISAM: (136293, 23)


## Nulos por columna y por año

Los nulos de SISAM vienen de municipios del panel que no están en SISAM o de semanas con menos de 7 días de dato.

In [5]:
anio = panel["semana"].dt.year
print("Nulos por columna:")
print(panel.isna().sum()[lambda s: s > 0])
print("\nNulos de SISAM por anio (pm2_5_reanalise como referencia):")
print(panel.groupby(anio)[COLS_SISAM + ["fuente_sisam"]].apply(lambda g: g.isna().sum()))
sin_sisam = panel.loc[panel["dias_sisam"] == 0, CLAVE].drop_duplicates()
print(f"\nMunicipios del panel sin ningun dia de SISAM: {len(sin_sisam)}")
if len(sin_sisam):
    print(sin_sisam.head(20).to_string(index=False))

Nulos por columna:
frp_promedio                97131
precipitacion_promedio      97600
dias_sin_lluvia_promedio    98813
riesgo_fuego_promedio       97977
riesgo_fuego_max            97977
dtype: int64

Nulos de SISAM por anio (pm2_5_reanalise como referencia):
        pm10_reanalise  pm2_5_reanalise  o3_reanalise  no2_reanalise  \
semana                                                                 
2022                 0                0             0              0   
2023                 0                0             0              0   
2024                 0                0             0              0   
2025                 0                0             0              0   
2026                 0                0             0              0   

        co_reanalise  so2_reanalise  fuente_sisam  
semana                                             
2022               0              0             0  
2023               0              0             0  
2024               0     

### Hallazgos de esta etapa

**Qué se observa**
- El panel pasó de **136,293** a **136,293** filas y de 15 a **23** columnas. No se perdieron filas.
- No hay nulos de SISAM en ningún año y **0** municipios del panel sin días de SISAM.
- Los nulos que quedan vienen de los focos. `frp_promedio` tiene **97,131** nulos, igual al número de semanas-municipio sin focos.

**Por qué se decidió así**
- El cruce es uno a uno por `(semana, municipio_norm, uf)`, y se verifica que no cambie el número de filas.


In [6]:
panel.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
print("Guardado:", RUTA_SALIDA.relative_to(BASE), panel.shape)
panel.head()

Guardado: datos/datasets_preprocesamiento/panel_semanal_sisam.csv (136293, 23)


,semana,municipio_norm,uf,lat,lon,num_incendios,frp_total,frp_promedio,precipitacion_promedio,dias_sin_lluvia_promedio,...,deter_area_ha,deter_num_alertas_area_protegida,pm10_reanalise,pm2_5_reanalise,o3_reanalise,no2_reanalise,co_reanalise,so2_reanalise,dias_sisam,fuente_sisam
0,2022-01-03,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,...,0.0,0,17.231429,13.100000,33.911429,0.474286,0.112857,0.372857,7,reanalise
1,2022-01-10,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,...,0.0,0,15.598571,12.035714,29.430000,0.412857,0.111429,0.370000,7,reanalise
2,2022-01-17,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,...,0.0,0,15.044286,11.148571,33.701429,0.468571,0.122857,0.388571,7,reanalise
3,2022-01-24,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,...,0.0,0,16.312857,12.434286,36.158571,0.461429,0.117143,0.350000,7,reanalise
4,2022-01-31,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,...,0.0,0,26.970000,21.907143,39.132857,0.440000,0.134286,0.354286,7,reanalise
